# After the Water: Pakistan Floods 2022
## Notebook B · Lab, Part B — When will the water be gone?

**How it works.** This lab is individual. Run the cells from the top (Shift+Enter). Where it says `TODO`, write your code, run the cell, then run the **test cell** under it and aim for all `[PASS]`. Cells marked **Given code**: read them, then run them.

**What's graded (30 points)**

- **Your code** (20 points): Tasks 1–5.
- **Two short write-ups** (10 points): each part ends with a class discussion, and **after each discussion you write about it**: a 2–3 sentence takeaway at the end of Part A, and a 5–8 sentence reflection at the end of Part B. 5 points each.

The questions and the *For the discussion* notes aren't graded, but they're how you prepare: the write-ups are much easier if you've answered them.

You need `district_peak.csv` and `helpers.py` in this folder. Keep the **Lab Part B Helper Sheet** open.

| Task | Function | What it does |
| --- | --- | --- |
| 4 | `daily_drop(series, days)` | how fast the water shrinks |
| 5 | `days_until_dry(series, days)` | the forecast |

## Where we are

Part B asks: when would the water be gone, and how far can we trust our answer? In Session 1 (Question 2) you guessed when Dadu would be dry. Now we make that forecast properly, then test it.

**Part B plan**

```
 85-100 min  Section 7     Task 4: how fast is the water shrinking?
100-115 min  Section 8     Task 5: when will it be dry?
115-135 min  Section 9     Test the forecast on the past
135-150 min  Section 10    Can we trust our answers? Two checks
150-165 min  DISCUSSION 2  Can we trust our answers? (whole class)
165-180 min  Your reflection (graded), then submit
```

In [1]:
import csv
from helpers import banner, check, summary

banner("READY")
print("banner() is loaded. Let's load the flood data again.")

PEAK_FILE = "district_peak.csv"

with open(PEAK_FILE, "r", encoding="utf-8", newline="") as f:
    districts = list(csv.DictReader(f))

print("Districts loaded:", len(districts), "  (expected 160)")

WINDOW_LABELS = ["01-31 Aug", "25-31 Aug", "01-07 Sep", "08-14 Sep", "15-21 Sep",
                 "26 Sep-02 Oct", "03-09 Oct", "11-17 Oct", "14-20 Nov"]
WEEKLY_LABELS = WINDOW_LABELS[1:]      # everything except "01-31 Aug"


# GIVEN: look up one district's row by name (the same as in Part A).
def find_district(districts, name):
    """Return the row in districts whose district column equals name, or None."""
    for row in districts:
        if row["district"] == name:
            return row
    return None


########################################################################
#  READY
########################################################################
banner() is loaded. Let's load the flood data again.
Districts loaded: 160   (expected 160)


## Section 7 — How fast is the water shrinking?

DAYS (given) holds the middle day of each weekly window, counted from 1 August 2022 (day 0): "25-31 Aug" is day 27, "14-20 Nov" is day 108. The gaps aren't equal: the satellites skipped some weeks.

In [2]:
banner("SECTION 7 — The weekly windows on a timeline")

DAYS = [27, 34, 41, 48, 59, 66, 74, 108]

for i in range(len(WEEKLY_LABELS)):
    print("  ", WEEKLY_LABELS[i].ljust(14), "day", DAYS[i])


########################################################################
#  SECTION 7 — The weekly windows on a timeline
########################################################################
   25-31 Aug      day 27
   01-07 Sep      day 34
   08-14 Sep      day 41
   15-21 Sep      day 48
   26 Sep-02 Oct  day 59
   03-09 Oct      day 66
   11-17 Oct      day 74
   14-20 Nov      day 108


## Given code — water_series(row)

water_series(row) gives back a district's flood water (km2) in the 8 weekly windows, as a list of floats. Every forecast in Part B starts from a list like this.

In [3]:
def water_series(row):
    """GIVEN: return the flood water (km2) in the 8 weekly windows, as floats."""
    series = []
    for label in WEEKLY_LABELS:
        series.append(float(row["flood_km2 " + label]))
    return series


banner("SECTION 7 — Dadu's water, week by week")

dadu = find_district(districts, "Dadu")
dadu_water = water_series(dadu)
for i in range(len(dadu_water)):
    print("  ", WEEKLY_LABELS[i].ljust(14), "day", str(DAYS[i]).rjust(3), ":",
          round(dadu_water[i]), "km2")


########################################################################
#  SECTION 7 — Dadu's water, week by week
########################################################################
   25-31 Aug      day  27 : 2597 km2
   01-07 Sep      day  34 : 2503 km2
   08-14 Sep      day  41 : 2409 km2
   15-21 Sep      day  48 : 2303 km2
   26 Sep-02 Oct  day  59 : 2005 km2
   03-09 Oct      day  66 : 1810 km2
   11-17 Oct      day  74 : 1700 km2
   14-20 Nov      day 108 : 1402 km2


## Task 4 — daily_drop(series, days)

The simplest forecast: assume the water keeps shrinking at its average speed so far.

average speed = (first value - last value) / (last day - first day)

**What to do**

Write daily_drop(series, days): the average km2 of water lost per day, from the first to the last value. (If the water grew, it's negative.)

**Useful tools**

a_list[0] is the first item; a_list[-1] is the last.

In [4]:
def daily_drop(series, days):
    """Return the average km2 lost per day, from the first to the last value."""
    # >>> TODO (Task 4): write your code here <<<
    return (series[0] - series[-1]) / (days[-1] - days[0])

In [5]:
# TEST — Task 4: daily_drop      (given code: don't change this cell)
# Run the cell above first, and again every time you change daily_drop.
results = []
print("  On made-up lists:")
results.append(check("daily_drop([100, 80, 50], [0, 5, 10])", daily_drop([100, 80, 50], [0, 5, 10]), 5.0))
results.append(check("daily_drop([10, 30], [0, 4])   (the water grew)", daily_drop([10, 30], [0, 4]), -5.0))
got = daily_drop([2597.073, 2503.227, 2408.904, 2303.034, 2005.382, 1809.77, 1700.107, 1402.125], DAYS)
print()
print("  On Dadu's real water series:")
results.append(check("round(daily_drop(Dadu, DAYS), 2)", None if got is None else round(got, 2), 14.75))
summary("Task 4", results)

  On made-up lists:
  [PASS] daily_drop([100, 80, 50], [0, 5, 10]) gave 5.0
  [PASS] daily_drop([10, 30], [0, 4])   (the water grew) gave -5.0

  On Dadu's real water series:
  [PASS] round(daily_drop(Dadu, DAYS), 2) gave 14.75

  Task 4: 3 of 3 tests passed.
  Well done! Move on to the next task.


In [6]:
banner("TASK 4 — How fast is Dadu's water shrinking?")

if daily_drop(dadu_water, DAYS) is not None:
    print("Dadu loses about", round(daily_drop(dadu_water, DAYS), 2), "km2 of water a day.")
else:
    print("Finish Task 4 first.")


########################################################################
#  TASK 4 — How fast is Dadu's water shrinking?
########################################################################
Dadu loses about 14.75 km2 of water a day.


## Section 8 — When will it be dry?

### Task 5 — days_until_dry(series, days)

```
days left = water left at the end (series[-1]) / daily_drop
```

**What to do**

Write days_until_dry(series, days): the number of days after the last window until the water is gone. If daily_drop is 0 or negative, the water isn't shrinking: return None.

In [17]:
def days_until_dry(series, days):
    """Return the days after the last window until the water is gone,
    or None if the water isn't shrinking."""
    # >>> TODO (Task 5): write your code here <<<
    # diff = days[-1] - days[0]
    x = daily_drop(series,days)
    if x is None or x <= 0:
        return None
    return series[-1] / x

In [18]:
# TEST — Task 5: days_until_dry      (given code: don't change this cell)
# Run the cell above first, and again every time you change days_until_dry.
# days_until_dry uses daily_drop, so Task 4 needs to pass first.
results = []
print("  On made-up lists:")
first = days_until_dry([100, 80, 50], [0, 5, 10])
results.append(check("days_until_dry([100, 80, 50], [0, 5, 10])", first, 10.0))
if first is not None:      # only once the function gives back numbers
    results.append(check("days_until_dry([10, 30], [0, 4])   (the water grew)",
                         days_until_dry([10, 30], [0, 4]), None))
got = days_until_dry([2597.073, 2503.227, 2408.904, 2303.034, 2005.382, 1809.77, 1700.107, 1402.125], DAYS)
print()
print("  On Dadu's real water series:")
results.append(check("round(days_until_dry(Dadu, DAYS))", None if got is None else round(got), 95))
summary("Task 5", results)

  On made-up lists:
  [PASS] days_until_dry([100, 80, 50], [0, 5, 10]) gave 10.0
  [PASS] days_until_dry([10, 30], [0, 4])   (the water grew) gave None

  On Dadu's real water series:
  [PASS] round(days_until_dry(Dadu, DAYS)) gave 95

  Task 5: 3 of 3 tests passed.
  Well done! Move on to the next task.


**Given code** — turn "days after the last window" into a real date.

In [19]:
banner("SECTION 8 — Dadu's forecast, as a date")

import datetime

DAY_ZERO = datetime.date(2022, 8, 1)


def dry_date(series, days):
    """GIVEN: the forecast date when the water is gone, or None."""
    left = days_until_dry(series, days)
    if left is None:
        return None
    return DAY_ZERO + datetime.timedelta(days=round(days[-1] + left))


if daily_drop(dadu_water, DAYS) is not None and days_until_dry(dadu_water, DAYS) is not None:
    print("Days after 17 Nov until Dadu is dry:", round(days_until_dry(dadu_water, DAYS)))
    print("Dadu will be dry on:", dry_date(dadu_water, DAYS))
else:
    print("Finish Tasks 4 and 5 first.")


########################################################################
#  SECTION 8 — Dadu's forecast, as a date
########################################################################
Days after 17 Nov until Dadu is dry: 95
Dadu will be dry on: 2023-02-20


**A choice we just made:** the water keeps shrinking at its average speed, a straight line from the first week to the last. Section 9 and Check 2 test it.

**Question 5:** Compare this with your guess in Session 1 (Question 2). What does this forecast assume about the water?

**My answer:** It assumes the water will continue shrinking at the same average daily rate until it completely disappears.

**Given code** — the forecast for every district with at least 50 km2 of water left in mid-November.

In [20]:
banner("SECTION 8 — When will each district be dry?")

ready_b = (daily_drop(dadu_water, DAYS) is not None
           and days_until_dry(dadu_water, DAYS) is not None)

END_OF_2022 = datetime.date(2022, 12, 31)

if not ready_b:
    print("Finish Tasks 4 and 5 first; this section will run once they work.")
else:
    forecasts = []
    never = []
    for row in districts:
        series = water_series(row)
        if series[-1] < 50:
            continue
        date = dry_date(series, DAYS)
        if date is None:
            never.append(row["district"])
        else:
            forecasts.append([date, row["district"], round(series[-1])])
    forecasts.sort()

    still_wet = 0
    for date, name, water in forecasts:
        mark = ""
        if date > END_OF_2022:
            mark = "  <- still under water at New Year"
            still_wet = still_wet + 1
        print("  ", str(date), name.ljust(22), str(water).rjust(6), "km2 in Nov", mark)

    print()
    print(still_wet, "districts would still be under water on 31 Dec 2022.")
    print("Water NOT shrinking (no forecast):", never)


########################################################################
#  SECTION 8 — When will each district be dry?
########################################################################
   2022-11-22 D. I. Khan                137 km2 in Nov 
   2022-11-26 Jhal Magsi                149 km2 in Nov 
   2022-11-26 Rajanpur                  215 km2 in Nov 
   2022-12-01 Dera Ghazi Khan           217 km2 in Nov 
   2022-12-03 Leiah                     119 km2 in Nov 
   2022-12-05 Mirpur Khas               363 km2 in Nov 
   2022-12-10 Jaffarabad                357 km2 in Nov 
   2022-12-11 Larkana                   420 km2 in Nov 
   2022-12-12 Jhang                     106 km2 in Nov 
   2022-12-18 Ghotki                    487 km2 in Nov 
   2022-12-21 Shaheed Benazir Abad      735 km2 in Nov 
   2022-12-23 Sukkur                    422 km2 in Nov 
   2022-12-24 Multan                     62 km2 in Nov 
   2022-12-24 Tando Muhammad Khan       244 km2 in Nov 
   2022-12-26 Badin   

**For the discussion:** find two of the hardest-hit districts from Part A in this list. What does it mean for people there if the water stays into 2023? And look at the bottom of the list: do you believe Faisalabad won't be dry until 2029?

**My notes:** People may continue facing displacement, damaged homes, health risks, and difficulties accessing clean water.No, because the forecast assumes a constant rate of decline, which may change over time.

## Section 9 — How good is the forecast? Test it on the past

A back-test pretends it's earlier than it really is:

1. Hide the last window (mid-November).
2. Forecast using only the first 7 windows, with your daily_drop.
3. Compare the prediction with what the satellite really saw.

If the forecast is bad at predicting a past we know, we shouldn't trust it much about a future we don't.

**Given code** — backtest_error(series, days): read it, then run it. It gives back predicted - actual. A negative error means the forecast predicted less water than there really was: it was too optimistic.

In [21]:
def backtest_error(series, days):
    """GIVEN: predict the last value from the first 7, and return predicted - actual."""
    drop = daily_drop(series[:7], days[:7])
    predicted = series[6] - drop * (days[7] - days[6])
    if predicted < 0:
        predicted = 0
    return predicted - series[7]


banner("SECTION 9 — Back-testing Dadu")

if daily_drop(dadu_water, DAYS) is not None and backtest_error(dadu_water, DAYS) is not None:
    error = backtest_error(dadu_water, DAYS)
    print("Water in Dadu, mid-November, really:", round(dadu_water[7]), "km2")
    print("The forecast from mid-October said:  ", round(dadu_water[7] + error), "km2")
    print("Error (predicted - actual):          ", round(error), "km2")
else:
    print("Finish Task 4 first.")


########################################################################
#  SECTION 9 — Back-testing Dadu
########################################################################
Water in Dadu, mid-November, really: 1402 km2
The forecast from mid-October said:   1051 km2
Error (predicted - actual):           -351 km2


**Given code** — back-test every district with at least 50 km2 of water in mid-October.

In [22]:
banner("SECTION 9 — Back-testing every district")

ready_c = ready_b and backtest_error(dadu_water, DAYS) is not None

if not ready_c:
    print("Finish Tasks 4-5 first; this section will run once they work.")
else:
    tested = 0
    too_optimistic = 0
    too_pessimistic = 0
    total_percent = 0
    for row in districts:
        series = water_series(row)
        if series[6] < 50:
            continue
        error = backtest_error(series, DAYS)
        percent = 100 * error / series[7]
        tested = tested + 1
        total_percent = total_percent + abs(percent)
        if error < 0:
            too_optimistic = too_optimistic + 1
        else:
            too_pessimistic = too_pessimistic + 1
        print("  ", row["district"].ljust(22), "predicted", str(round(series[7] + error)).rjust(6),
              " actual", str(round(series[7])).rjust(6), " error", str(round(percent)).rjust(5), "%")

    print()
    print("Districts tested:", tested)
    print("  Forecast predicted TOO LITTLE water (too optimistic):", too_optimistic)
    print("  Forecast predicted TOO MUCH water (too pessimistic): ", too_pessimistic)
    print("  Average size of the error:", round(total_percent / tested), "%")


########################################################################
#  SECTION 9 — Back-testing every district
########################################################################
   Awaran                 predicted    118  actual    143  error   -18 %
   Duki                   predicted     13  actual     55  error   -76 %
   Gwadar                 predicted    295  actual    260  error    13 %
   Jaffarabad             predicted      0  actual    357  error  -100 %
   Jhal Magsi             predicted      0  actual    149  error  -100 %
   Kech                   predicted     43  actual     70  error   -39 %
   Khuzdar                predicted     90  actual     68  error    32 %
   Lasbela                predicted    255  actual    283  error   -10 %
   Panjgur                predicted      0  actual     24  error  -100 %
   Sohbatpur              predicted    145  actual    255  error   -43 %
   D. I. Khan             predicted      0  actual    137  error  -100 %
   Hari

**Question 6:** Was the forecast usually too optimistic or too pessimistic? So are our dry dates in Section 8 probably too early, or too late?

**My answer:** It was too optimistic in 44 out of 54 districts, meaning the predicted dry dates are probably earlier than reality.

**For the discussion:** pick one district where the back-test went badly wrong and look at its 8 water values. What did the water do that a straight line can't capture?

**My notes:** The water levels may have fluctuated, increased, or decreased at different rates instead of following a constant pattern.

## Section 10 — Can we trust our answers?

Every answer in this case study rested on a choice: which window counts as "hit hardest", what counts as "help", how to draw the forecast line. To test an answer, change one choice and see if the answer survives. No new code here: just run each cell and think.

### Check 1 — "The 15 hardest-hit districts are all in Sindh."

What if we rank by the whole-month August window instead of the weekly peak?

In [23]:
banner("SECTION 10a — Check 1: the top 15, measured two ways")


def top_15(districts, peak_of):
    """GIVEN: the names of the 15 districts with the biggest peak_of(row)."""
    ranked = []
    for row in districts:
        ranked.append([peak_of(row), row["district"]])
    ranked.sort(reverse=True)
    names = []
    for value, name in ranked[:15]:
        names.append(name)
    return names


def weekly_peak(row):
    """GIVEN: the most people exposed in any weekly window (Part A, Task 3)."""
    values = []
    for label in WEEKLY_LABELS:
        values.append(float(row["exposed " + label]))
    return max(values)


def august_peak(row):
    """GIVEN: the people exposed in the whole-month 01-31 Aug window."""
    return float(row["exposed 01-31 Aug"])


by_week = top_15(districts, weekly_peak)
by_august = top_15(districts, august_peak)

province_of = {}
for row in districts:
    province_of[row["district"]] = row["province"]

print("Top 15 by WEEKLY peak:")
for name in by_week:
    print("  ", name.ljust(22), province_of[name])
print()
print("Entered the top 15 with the AUGUST window:")
for name in by_august:
    if name not in by_week:
        print("  ", name.ljust(22), province_of[name])
print("Left the top 15 with the AUGUST window:")
for name in by_week:
    if name not in by_august:
        print("  ", name.ljust(22), province_of[name])


########################################################################
#  SECTION 10a — Check 1: the top 15, measured two ways
########################################################################
Top 15 by WEEKLY peak:
   Khairpur               Sindh
   Kambar Shahdad Kot     Sindh
   Larkana                Sindh
   Dadu                   Sindh
   Shikarpur              Sindh
   Naushahro Feroze       Sindh
   Jacobabad              Sindh
   Shaheed Benazir Abad   Sindh
   Badin                  Sindh
   Sanghar                Sindh
   Mirpur Khas            Sindh
   Kashmore               Sindh
   Hyderabad              Sindh
   Sukkur                 Sindh
   Ghotki                 Sindh

Entered the top 15 with the AUGUST window:
   Sialkot                Punjab
   Gujranwala             Punjab
Left the top 15 with the AUGUST window:
   Hyderabad              Sindh
   Sukkur                 Sindh


**Question 7:** Does "the 15 hardest-hit districts are all in Sindh" survive the change? Which measure would you trust more, and why?

**My answer:** No, because Sialkot and Gujranwala entered the top 15 while Hyderabad and Sukkur left; I would trust weekly peaks because they capture the worst exposure period.

### Check 2 — "Dadu will be dry on 20 February 2023."

Our forecast drew one straight line from the first week to the last. What if we use only the last 2 weeks? This forecasts Dadu that way, and back-tests the choice on every district.

In [24]:
banner("SECTION 10b — Check 2: a forecast from the last 2 weeks only")

if not ready_c:
    print("Finish Tasks 4-5 first; this section will run once they work.")
else:
    dadu_last2 = dry_date(dadu_water[-2:], DAYS[-2:])
    print("Dadu, straight line over all 8 weeks:", dry_date(dadu_water, DAYS))
    print("Dadu, straight line over the last 2: ", dadu_last2)
    print()

    tested = 0
    optimistic_all = 0
    optimistic_last2 = 0
    percent_all = 0
    percent_last2 = 0
    for row in districts:
        series = water_series(row)
        if series[6] < 50:
            continue
        tested = tested + 1
        # all 7 weeks before mid-November (backtest_error, Section 9)
        error = backtest_error(series, DAYS)
        percent_all = percent_all + abs(100 * error / series[7])
        if error < 0:
            optimistic_all = optimistic_all + 1
        # only the last 2 weeks before mid-November
        drop = daily_drop(series[5:7], DAYS[5:7])
        predicted = max(0, series[6] - drop * (DAYS[7] - DAYS[6]))
        error = predicted - series[7]
        percent_last2 = percent_last2 + abs(100 * error / series[7])
        if error < 0:
            optimistic_last2 = optimistic_last2 + 1

    print("Back-test on", tested, "districts:")
    print("   all 7 weeks:  average error", round(percent_all / tested), "%,",
          "too optimistic in", optimistic_all)
    print("   last 2 weeks: average error", round(percent_last2 / tested), "%,",
          "too optimistic in", optimistic_last2)


########################################################################
#  SECTION 10b — Check 2: a forecast from the last 2 weeks only
########################################################################
Dadu, straight line over all 8 weeks: 2023-02-20
Dadu, straight line over the last 2:  2023-04-26

Back-test on 54 districts:
   all 7 weeks:  average error 59 %, too optimistic in 44
   last 2 weeks: average error 44 %, too optimistic in 35


**Question 8:** Which choice gave the better back-test? Is even the better one good enough to plan with? Use the numbers.

**My answer:** The last-two-weeks method performed better, with 44% average error compared to 59%, but it is still too inaccurate for major planning decisions.

## Discussion 2 — Can we trust our answers?

Stop here: we'll talk about Part B, and the whole case study, as a class (about 15 minutes). Bring your answers and notes:

1. what it means for people if the water stays into 2023 (your note)
2. Faisalabad dry in 2029: when should we not trust a forecast? (your note)
3. a back-test that went badly wrong, and why (your note)
4. which findings survived a check, and which didn't (Questions 7-8)
5. what you'd tell someone deciding where to send help

**This is graded (5 points):** after the discussion, write your reflection in the next cell.

## Your reflection (graded, 5 points)

Write 5 to 8 sentences, after the discussion. Pick ONE finding from this case study. Here are some, with the choice each one rested on:

```
Finding                                 The choice behind it                Where
Bagh is in Azad Kashmir                 trusting the order of the rows      Session 1
Islamabad: one name, two places         position over name                  Session 1
Musakhel: 188% of people exposed        the whole-August window             Session 2
KP's districts don't add up to KP       a 1% threshold                      Session 2
46.6 million "people reached"           adding people up across records     Part A
Sindh: 72% of the need, 62% of records  "help" = records; share vs need     Part A
All 15 hardest hit are in Sindh         weekly peak, not August             Part A, Check 1
Dadu dry by 20 Feb 2023                 a straight line over all weeks      Part B, Check 2
```

Then say:

1. what we found, and the choice it rested on,
2. what changed when you looked again (or what could change it),
3. how sure you are of it now, and why, and
4. what someone deciding where to send help should know about it.

**My answer:** 

It means prolonged displacement, financial difficulties, health problems, and continued dependence on relief.
We should question forecasts that extend too far into the future or rely on assumptions that may not remain true.
Jaffarabad had a 100% error because the forecast predicted zero water while the actual water level was 357 km².
The Sindh-only finding did not survive the August comparison, while the last-two-weeks forecast performed better than the all-weeks forecast.
Prioritize districts with high water exposure and ongoing needs rather than relying only on predicted dry dates.

## Wrap-up

The big idea: what we found doesn't make the data useless. It changes what the data can honestly say.

Before you submit: restart each notebook and run all its cells.

In [25]:
banner("END OF THE LAB")


########################################################################
#  END OF THE LAB
########################################################################
